# Confronto: U-Net Idaho (dominio ristretto) vs U-Net Extended (3-state)

**Ipotesi**: la U-Net performa meglio quando allenata su un dominio ristretto (Idaho/SPORTLIS) rispetto a un dominio esteso (3 stati).

## Modelli a confronto

| | **Modello A — Idaho** | **Modello B — Extended** |
|---|---|---|
| Notebook | `sportlis_unet_NARR_TM_lean_loyo_TPI_canopy.ipynb` | `sportlis_unet_extended_1985_2024.ipynb` |
| Griglia | SPORTLIS (243×223, ~3km) | Extended (484×698, ~3km) |
| Dominio | Idaho: lat [41.8-49.1°N], lon [-117.5/-110.8°E] | 3 stati: WA/OR/ID/CA/NV/CO |
| Anni training | 2017-2021 (LOYO) | 1985-2024 (LOYO step=5) |
| N fold test | 5 | 8 |
| Canali input | 16 (7 TM + 2 latlon + 7 topo) | 16 (identici) |
| Architettura | UNet DoubleConv+GN+GELU, base=32 | identica |

**Anno sovrapposto (out-of-sample per entrambi): 2020**

## Struttura del notebook
1. Metriche aggregate per modello
2. Confronto diretto anno 2020
3. XAI: feature importance a confronto
4. Conclusioni

In [ ]:
# 0. IMPORTS
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
# Adatta PROJECT_ROOT al tuo ambiente
import os
if Path('/Users/irene/PROJECTS.index/Hydrology/projects/sportlis_project').exists():
    PROJECT_ROOT = Path('/Users/irene/PROJECTS.index/Hydrology/projects/sportlis_project')
elif Path('/glade/work').exists():
    # NCAR
    PROJECT_ROOT = Path('/glade/work') / os.environ.get('USER', 'irene') / 'sportlis_project'
else:
    PROJECT_ROOT = Path('.').resolve()

NOTEBOOK_DIR = Path('.').resolve()  # cartella con i notebook

# ── Idaho model outputs ────────────────────────────────────────────────────────
IDAHO_OUT = NOTEBOOK_DIR / 'sportlis_unet_NARR_lean_TPI_canopy_loyo_outputs'
IDAHO_FOLD_CSV  = IDAHO_OUT / 'loyo_lean_TPI_canopy_fold_metrics.csv'
IDAHO_XAI_CSV   = IDAHO_OUT / 'xai_loyo_lean_TPI_canopy_importance.csv'
IDAHO_XAI_NPZ   = IDAHO_OUT / 'xai_loyo_lean_TPI_canopy_importance.npz'

# ── Extended model outputs ─────────────────────────────────────────────────────
EXT_OUT = NOTEBOOK_DIR / 'sportlis_unet_extended_loyo_outputs'
EXT_BASIN_CSV   = NOTEBOOK_DIR / 'basin_ts_results.csv'          # generato da analysis_extended_basins.ipynb
EXT_XAI_BASIN   = NOTEBOOK_DIR / 'xai_basin_importance.npz'     # generato da analysis_extended_basins.ipynb
EXT_FOLD_CSV    = EXT_OUT / 'extended_fold_metrics.csv'          # se esiste

# ── Channel names (stessa architettura) ──────────────────────────────────────
CH_NAMES_TM   = ['precip_7d', 'precip_30d', 'precip_60d', 'precip_wytd',
                  'tair_30d_mean', 'tair_30d_max', 'degree_day_30d']
CH_NAMES_TOPO = ['elevation', 'slope', 'aspect_sin', 'aspect_cos',
                  'tpi_short', 'tpi_long', 'canopy_fraction']
CH_NAMES_ALL  = CH_NAMES_TM + ['lat_norm', 'lon_norm'] + CH_NAMES_TOPO  # 16 totali

print('Paths set up.')
print(f'  Idaho fold CSV  : {IDAHO_FOLD_CSV}  exists={IDAHO_FOLD_CSV.exists()}')
print(f'  Idaho XAI CSV   : {IDAHO_XAI_CSV}   exists={IDAHO_XAI_CSV.exists()}')
print(f'  Ext basin CSV   : {EXT_BASIN_CSV}   exists={EXT_BASIN_CSV.exists()}')
print(f'  Ext XAI basin   : {EXT_XAI_BASIN}   exists={EXT_XAI_BASIN.exists()}')

In [ ]:
# 1. CARICA METRICHE FOLD — Idaho model
df_idaho = pd.read_csv(IDAHO_FOLD_CSV)
df_idaho['model'] = 'Idaho'
print('Idaho model — fold metrics:')
display(df_idaho[['test_year','mae','bias','rmse','r2','nse','bm_pearson_r']].round(3))
print(f"\nMedia aggregata: MAE={df_idaho.mae.mean():.2f} mm  BIAS={df_idaho.bias.mean():+.2f}  "
      f"R²={df_idaho.r2.mean():.3f}  NSE={df_idaho.nse.mean():.3f}")

In [ ]:
# 2. CARICA RISULTATI EXTENDED — basin_ts_results.csv, solo Idaho basin
#
# Se basin_ts_results.csv non esiste ancora (notebook non ancora girato su NCAR)
# questo cell stampa un avviso e continua con quello che ha.

df_ext_idaho = None
if EXT_BASIN_CSV.exists():
    df_basin = pd.read_csv(EXT_BASIN_CSV)
    print(f'Basin TS CSV loaded: {len(df_basin)} rows, columns: {list(df_basin.columns)}')
    print(f'Basins available: {sorted(df_basin["basin"].unique())}')

    df_ext_idaho = df_basin[df_basin['basin'] == 'Idaho'].copy()
    df_ext_idaho['model'] = 'Extended'

    # Rinomina colonne per uniformità con Idaho model
    # Attese da cell 10 di analysis_extended_basins.ipynb:
    # year, basin, obs_mean, pred_mean, mae, bias
    print(f'\nExtended model — Idaho basin ({len(df_ext_idaho)} test years):')
    display(df_ext_idaho.round(3))

    if len(df_ext_idaho) > 0:
        print(f"\nMedia aggregata Extended@Idaho: "
              f"MAE={df_ext_idaho['mae'].mean():.2f} mm  "
              f"BIAS={df_ext_idaho['bias'].mean():+.2f} mm")
else:
    print("⚠️  basin_ts_results.csv non trovato.")
    print("   Esegui prima analysis_extended_basins.ipynb su NCAR (cell 10 — Basin Time Series).")
    print("   Il confronto Extended model sarà parziale.")

In [ ]:
# 3. SUMMARY TABLE — confronto aggregato
#
# Domande chiave:
#   (a) Modello A (Idaho, 2017-2021) vs Modello B (Extended, 1985-2024) su Idaho pixels
#   (b) Anno 2020: unico anno test sovrapposto out-of-sample per entrambi

print('=' * 65)
print('  CONFRONTO AGGREGATO — Idaho pixels')
print('=' * 65)
print(f'{"Modello":25s} {"N_folds":>8} {"MAE [mm]":>10} {"BIAS [mm]":>10} {"RMSE":>8} {"R²":>6}')
print('-' * 65)

# Idaho model (global metrics, tutti i pixel SPORTLIS = Idaho)
print(f'{"Idaho (2017-2021)":25s} '
      f'{len(df_idaho):>8} '
      f'{df_idaho.mae.mean():>10.2f} '
      f'{df_idaho.bias.mean():>+10.2f} '
      f'{df_idaho.rmse.mean():>8.2f} '
      f'{df_idaho.r2.mean():>6.3f}')

if df_ext_idaho is not None and len(df_ext_idaho) > 0:
    ext_cols = df_ext_idaho.columns.tolist()
    mae_col  = 'mae'  if 'mae'  in ext_cols else [c for c in ext_cols if 'mae'  in c.lower()][0]
    bias_col = 'bias' if 'bias' in ext_cols else [c for c in ext_cols if 'bias' in c.lower()][0]
    rmse_col = 'rmse' if 'rmse' in ext_cols else None
    r2_col   = 'r2'   if 'r2'   in ext_cols else None

    print(f'{"Extended (1985-2024)@Idaho":25s} '
          f'{len(df_ext_idaho):>8} '
          f'{df_ext_idaho[mae_col].mean():>10.2f} '
          f'{df_ext_idaho[bias_col].mean():>+10.2f} '
          f'{df_ext_idaho[rmse_col].mean():>8.2f} ' if rmse_col else '         -  '
          f'{df_ext_idaho[r2_col].mean():>6.3f}' if r2_col else '     -')
else:
    print(f'{"Extended (1985-2024)@Idaho":25s} {"N/A — run NCAR first":>45}')

print('=' * 65)

print()
print('  Anno sovrapposto out-of-sample: 2020')
print('-' * 65)
row_2020_idaho = df_idaho[df_idaho.test_year == 2020]
if len(row_2020_idaho) > 0:
    r = row_2020_idaho.iloc[0]
    print(f'{"Idaho @ 2020":25s} MAE={r.mae:6.2f} mm  BIAS={r.bias:+6.2f} mm  R²={r.r2:.3f}')

if df_ext_idaho is not None:
    row_2020_ext = df_ext_idaho[df_ext_idaho.year == 2020]
    if len(row_2020_ext) > 0:
        r = row_2020_ext.iloc[0]
        print(f'{"Extended@Idaho @ 2020":25s} MAE={r[mae_col]:6.2f} mm  BIAS={r[bias_col]:+6.2f} mm')
    else:
        print(f'{"Extended@Idaho @ 2020":25s} anno 2020 non ancora computato')
print('=' * 65)

In [ ]:
# 4. FIGURA 1 — confronto MAE per anno test
#
# Plot a barre affiancate: Idaho model (blu) vs Extended@Idaho (arancio)
# Solo per anni disponibili. Evidenziato 2020 (anno sovrapposto).

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: MAE per fold ────────────────────────────────────────────────────────
ax = axes[0]
ax.bar([str(y) for y in df_idaho.test_year], df_idaho.mae,
       color='#4C72B0', edgecolor='k', alpha=0.85, label='Idaho model (2017-2021)')
for i, (y, v) in enumerate(zip(df_idaho.test_year, df_idaho.mae)):
    ax.text(i, v + 0.2, f'{v:.1f}', ha='center', fontsize=8)

if df_ext_idaho is not None and len(df_ext_idaho) > 0:
    ext_years_str = [str(int(y)) for y in df_ext_idaho.year]
    ax.bar(ext_years_str, df_ext_idaho[mae_col],
           color='#DD8452', edgecolor='k', alpha=0.85, label='Extended@Idaho (1985-2024)',
           width=0.4, align='edge')

# Evidenzia 2020
ax.axvline(x=df_idaho[df_idaho.test_year==2020].index[0] if 2020 in df_idaho.test_year.values else -1,
           color='red', lw=1.5, ls='--', alpha=0.5, label='2020 (anno sovrapposto)')
ax.set_xlabel('Anno test'); ax.set_ylabel('MAE [mm]')
ax.set_title('MAE per fold — Idaho vs Extended@Idaho')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

# ── Right: BIAS per fold ───────────────────────────────────────────────────────
ax = axes[1]
ax.bar([str(y) for y in df_idaho.test_year], df_idaho.bias,
       color='#4C72B0', edgecolor='k', alpha=0.85, label='Idaho model')
for i, (y, v) in enumerate(zip(df_idaho.test_year, df_idaho.bias)):
    ax.text(i, v + (0.2 if v >= 0 else -0.6), f'{v:+.1f}', ha='center', fontsize=8)

if df_ext_idaho is not None and len(df_ext_idaho) > 0:
    ax.bar(ext_years_str, df_ext_idaho[bias_col],
           color='#DD8452', edgecolor='k', alpha=0.85, label='Extended@Idaho',
           width=0.4, align='edge')

ax.axhline(0, color='k', lw=0.8)
ax.set_xlabel('Anno test'); ax.set_ylabel('BIAS [mm]  (pred − obs)')
ax.set_title('BIAS per fold — Idaho vs Extended@Idaho')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_mae_bias.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: comparison_mae_bias.png')

In [ ]:
# 5. CARICA XAI — Idaho model
#
# xai_loyo_lean_TPI_canopy_importance.csv:
#   rows = test folds (test_2017 ... test_2021)
#   cols = channel names
#   values = ΔMAE [mm] (positivo = canale importante)

df_xai_idaho = pd.read_csv(IDAHO_XAI_CSV, index_col=0)
print('Idaho XAI shape:', df_xai_idaho.shape)
print(df_xai_idaho.round(2))

# Media e std su fold
xai_idaho_mean = df_xai_idaho.mean(axis=0)    # Series: channel -> ΔMAE medio
xai_idaho_std  = df_xai_idaho.std(axis=0)

print('\nTop-10 channels (Idaho model, mean ΔMAE):')
top10 = xai_idaho_mean.sort_values(ascending=False).head(10)
for ch, v in top10.items():
    print(f'  {ch:22s}: {v:+.3f} mm')

In [ ]:
# 6. CARICA XAI — Extended model (per basin)
#
# xai_basin_importance.npz (da analysis_extended_basins.ipynb cell 8):
#   imp_basin : (n_basins, n_folds, n_ch)  ΔMAE per basin/fold/channel
#   mae_base_all: (n_basins, n_folds)
#   done_folds: list of done fold test years

BASINS_ORDER = [
    'Pacific NW Cascades', 'Oregon Cascades', 'Sierra Nevada',
    'Northern Rockies',   'Southern Rockies (CO)', 'Idaho'
]

xai_ext_idaho_mean = None
mae_baseline_ext_idaho = None

if EXT_XAI_BASIN.exists():
    _z = np.load(str(EXT_XAI_BASIN), allow_pickle=True)
    imp_basin    = _z['imp_basin']    # (n_basins, n_folds, n_ch)
    mae_base_all = _z['mae_base_all'] # (n_basins, n_folds)
    done_folds   = list(_z['done_folds'])
    print(f'Ext XAI loaded: imp_basin shape={imp_basin.shape}, done_folds={done_folds}')

    n_ch_ext = imp_basin.shape[-1]
    n_basins_ext = imp_basin.shape[0]
    CH_EXT = CH_NAMES_ALL[:n_ch_ext]  # assume stessa architettura

    # Trova indice basin Idaho
    try:
        idaho_bidx = BASINS_ORDER.index('Idaho')
    except ValueError:
        idaho_bidx = n_basins_ext - 1  # ultimo basin per default
        print(f'  WARNING: Idaho non trovato in BASINS_ORDER, uso idx={idaho_bidx}')

    # Media su fold (esclude fold con ΔMAE = 0 = non completati)
    imp_idaho_folds = imp_basin[idaho_bidx]           # (n_folds, n_ch)
    valid_fold_mask = np.abs(imp_idaho_folds).sum(axis=1) > 0
    if valid_fold_mask.sum() > 0:
        xai_ext_idaho_mean  = imp_idaho_folds[valid_fold_mask].mean(axis=0)  # (n_ch,)
        mae_baseline_ext_idaho = mae_base_all[idaho_bidx][valid_fold_mask].mean()
        print(f'  Idaho basin: {valid_fold_mask.sum()}/{len(done_folds)} folds validi')
        print(f'  MAE baseline Extended@Idaho: {mae_baseline_ext_idaho:.2f} mm')
    else:
        print('  ⚠️  Idaho basin: nessun fold valido nel npz. Rilanciare XAI su NCAR.')
else:
    print('⚠️  xai_basin_importance.npz non trovato.')
    print('   Esegui analysis_extended_basins.ipynb (cell 8 — XAI Basin Compute) su NCAR.')

In [ ]:
# 7. FIGURA 2 — XAI side-by-side: Idaho model vs Extended@Idaho
#
# Layout: due pannelli affiancati, stesso asse y (ΔMAE mm) o
# ΔMAE / MAE_baseline * 100 (importanza relativa %) per confronto equo
# dato che i MAE baseline sono diversi (10.13mm vs ?).

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)
colors = ['#4C72B0', '#DD8452']

def plot_xai_bar(ax, ch_means, ch_stds, ch_names, baseline_mae, color, title, top_n=12):
    """Barplot ΔMAE (assoluto e % del baseline)."""
    # Ordina per importanza decrescente
    idx_sorted = np.argsort(ch_means)[::-1][:top_n]
    vals  = ch_means[idx_sorted]
    errs  = ch_stds[idx_sorted] if ch_stds is not None else np.zeros_like(vals)
    names = [ch_names[i] for i in idx_sorted]

    y_pos = np.arange(len(vals))
    ax.barh(y_pos, vals, xerr=errs, color=color, edgecolor='k',
            alpha=0.85, error_kw={'capsize': 3, 'lw': 1.2})
    for j, (v, n) in enumerate(zip(vals, names)):
        pct = v / baseline_mae * 100
        ax.text(v + max(vals) * 0.01, j, f' {v:.2f} mm ({pct:.0f}%)',
                va='center', fontsize=8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel('ΔMAE [mm]  (permutation importance)')
    ax.set_title(f'{title}\n(baseline MAE = {baseline_mae:.2f} mm)', fontsize=10)
    ax.axvline(0, color='k', lw=0.8)
    ax.grid(axis='x', alpha=0.3)

# ── Idaho model XAI ───────────────────────────────────────────────────────────
ch_arr_idaho = np.array([xai_idaho_mean.get(c, 0.0) for c in CH_NAMES_ALL])
ch_std_idaho = np.array([xai_idaho_std.get(c, 0.0) for c in CH_NAMES_ALL])
mae_bl_idaho = df_idaho.mae.mean()

plot_xai_bar(axes[0], ch_arr_idaho, ch_std_idaho, CH_NAMES_ALL,
             mae_bl_idaho, colors[0], 'Idaho model (2017-2021)\nSPORTLIS 243×223 grid')

# ── Extended@Idaho XAI ────────────────────────────────────────────────────────
if xai_ext_idaho_mean is not None:
    ch_std_ext = imp_idaho_folds[valid_fold_mask].std(axis=0)
    plot_xai_bar(axes[1], xai_ext_idaho_mean, ch_std_ext, CH_EXT,
                 mae_baseline_ext_idaho, colors[1],
                 'Extended model @ Idaho basin (1985-2024)\nExtended 484×698 grid, Idaho pixels')
else:
    axes[1].text(0.5, 0.5, 'XAI Extended not available yet.\nRun analysis_extended_basins.ipynb on NCAR.',
                 ha='center', va='center', transform=axes[1].transAxes, fontsize=11, color='gray')
    axes[1].set_title('Extended model @ Idaho basin\n(pending NCAR run)', fontsize=10)

plt.suptitle('Feature Importance (XAI permutation) — Idaho vs Extended@Idaho', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('comparison_xai.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: comparison_xai.png')

In [ ]:
# 8. FIGURA 3 — Confronto per anno 2020 (unico anno sovrapposto)
#
# Left: MAE/BIAS/R² side-by-side per 2020
# Right: XAI ranking shifts (scatter: Idaho rank vs Extended rank)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: metriche 2020 ───────────────────────────────────────────────────────
ax = axes[0]
metrics_list = []

row_idaho_2020 = df_idaho[df_idaho.test_year == 2020].iloc[0]
metrics_list.append({
    'model': 'Idaho model\n(2020 fold)',
    'MAE': row_idaho_2020.mae,
    'BIAS': row_idaho_2020.bias,
    'R²': row_idaho_2020.r2,
    'NSE': row_idaho_2020.nse,
})

if df_ext_idaho is not None and len(df_ext_idaho) > 0:
    row_ext_2020 = df_ext_idaho[df_ext_idaho.year == 2020]
    if len(row_ext_2020) > 0:
        r = row_ext_2020.iloc[0]
        row_dict = {
            'model': 'Extended@Idaho\n(2020 fold)',
            'MAE': r[mae_col],
            'BIAS': r[bias_col],
        }
        if rmse_col: row_dict['RMSE'] = r[rmse_col]
        if r2_col:   row_dict['R²']   = r[r2_col]
        metrics_list.append(row_dict)

metric_names = ['MAE', 'BIAS']  # add R² if available
if all('R²' in m for m in metrics_list): metric_names.append('R²')

x = np.arange(len(metric_names))
width = 0.35
for i, (m, c) in enumerate(zip(metrics_list, colors)):
    vals_plot = [m.get(k, np.nan) for k in metric_names]
    bars = ax.bar(x + i*width - width/2, vals_plot, width, label=m['model'],
                  color=c, edgecolor='k', alpha=0.85)
    for bar, v in zip(bars, vals_plot):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1*(1 if v >= 0 else -1),
                    f'{v:+.2f}' if 'BIAS' in metric_names else f'{v:.2f}',
                    ha='center', fontsize=8, rotation=0)
ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(metric_names)
ax.set_title('Confronto metriche — Anno 2020\n(unico test year sovrapposto)', fontsize=10)
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# ── Right: XAI rank comparison ────────────────────────────────────────────────
ax = axes[1]

# Rank Idaho
rank_idaho = {ch: r+1 for r, ch in enumerate(
    sorted(CH_NAMES_ALL, key=lambda c: -xai_idaho_mean.get(c, 0.0))
)}

if xai_ext_idaho_mean is not None:
    ch_ext_dict = dict(zip(CH_EXT, xai_ext_idaho_mean))
    rank_ext = {ch: r+1 for r, ch in enumerate(
        sorted(CH_EXT, key=lambda c: -ch_ext_dict.get(c, 0.0))
    )}

    common_chs = [c for c in CH_NAMES_ALL if c in rank_ext]
    rx = [rank_idaho[c] for c in common_chs]
    ry = [rank_ext[c]   for c in common_chs]

    ax.scatter(rx, ry, s=60, zorder=3, color='#555')
    for c, xi, yi in zip(common_chs, rx, ry):
        delta = abs(xi - yi)
        fc = 'red' if delta >= 5 else ('orange' if delta >= 2 else 'gray')
        ax.annotate(c, (xi, yi), fontsize=7.5, color=fc,
                    xytext=(4, 4), textcoords='offset points')
    ax.plot([1, 16], [1, 16], 'k--', lw=0.8, alpha=0.5, label='rank invariato')
    ax.set_xlabel('Rank Idaho model (1=più importante)')
    ax.set_ylabel('Rank Extended@Idaho')
    ax.set_title('XAI rank shift: Idaho vs Extended@Idaho\n(rosso = shift ≥ 5 posizioni)', fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_xlim(0, 17); ax.set_ylim(0, 17)
else:
    ax.text(0.5, 0.5, 'XAI Extended not available yet.',
            ha='center', va='center', transform=ax.transAxes, fontsize=11, color='gray')
    ax.set_title('XAI rank shift (pending NCAR run)', fontsize=10)

plt.tight_layout()
plt.savefig('comparison_2020_xai_rank.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: comparison_2020_xai_rank.png')

In [ ]:
# 9. FIGURA 4 — Time series SWE basin-mean confronto (se disponibili)
#
# Idaho model: usa basin_mean_per_fold (dalla cella 14 del notebook Idaho,
#              non serializzato → qui usiamo solo i valori aggregati CSV)
# Extended: usa colonne obs_mean/pred_mean di basin_ts_results.csv

if df_ext_idaho is not None and 'obs_mean' in df_ext_idaho.columns and 'pred_mean' in df_ext_idaho.columns:
    fig, ax = plt.subplots(figsize=(14, 5))

    # Extended — time series SWE basin-mean per anno test
    # Le colonne obs_mean e pred_mean sono scalari per anno (mean su tutti i timestep snowy)
    years_ext = df_ext_idaho.year.values.astype(int)
    obs_ext   = df_ext_idaho.obs_mean.values
    pred_ext  = df_ext_idaho.pred_mean.values

    ax.plot(years_ext, obs_ext,  'ko-', label='SPORTLIS obs (Extended folds)', lw=1.5, ms=6)
    ax.plot(years_ext, pred_ext, 's--', color='#DD8452', label='Extended@Idaho pred', lw=1.5, ms=6)

    # Idaho model fold MAE come error bars sul plot (approx)
    # usa test_year vs (obs_mean = true_mean, pred_mean = true_mean + bias)
    # Non abbiamo i valori assoluti per l'Idaho model in CSV → solo metriche
    # Aggiunge annotazioni MAE
    for _, row in df_idaho.iterrows():
        ax.annotate(f'Idaho\nMAE={row.mae:.1f}mm',
                    xy=(row.test_year, obs_ext[np.argmin(np.abs(years_ext - row.test_year))] if len(obs_ext) > 0 else 0),
                    xytext=(0, 25), textcoords='offset points',
                    ha='center', fontsize=7, color='#4C72B0',
                    arrowprops=dict(arrowstyle='->', color='#4C72B0', lw=0.8))

    ax.set_xlabel('Anno test'); ax.set_ylabel('Basin-mean SWE [mm]')
    ax.set_title('SWE basin-mean Idaho pixels — Extended model (obs vs pred)\n'
                 'con annotazioni MAE Idaho model per confronto')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('comparison_ts_idaho.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: comparison_ts_idaho.png')
else:
    print('Time series plot: basin_ts_results.csv non disponibile o mancano colonne obs_mean/pred_mean.')
    print('Sarà disponibile dopo run su NCAR.')

In [ ]:
# 10. CONCLUSIONI — Stampa automatica interpretazione risultati
#
# Risponde all'ipotesi: dominio ristretto → migliori risultati?

print('=' * 70)
print('  INTERPRETAZIONE RISULTATI')
print('=' * 70)

mae_idaho = df_idaho.mae.mean()
print(f'\nModello Idaho (dominio ristretto, 2017-2021):')
print(f'  MAE medio = {mae_idaho:.2f} mm')
print(f'  XAI #1 = {xai_idaho_mean.idxmax()} ({xai_idaho_mean.max():.2f} mm, '
      f'{xai_idaho_mean.max()/mae_idaho*100:.0f}% del baseline)')

if df_ext_idaho is not None and len(df_ext_idaho) > 0:
    mae_ext = df_ext_idaho[mae_col].mean()
    print(f'\nModello Extended (dominio 3-state, 1985-2024) su Idaho pixels:')
    print(f'  MAE medio = {mae_ext:.2f} mm')

    delta_mae = mae_ext - mae_idaho
    winner = 'Idaho (dominio ristretto)' if delta_mae > 0 else 'Extended (dominio largo, più anni)'
    print(f'\nΔMAE (Extended - Idaho) = {delta_mae:+.2f} mm')
    print(f'Il modello migliore su Idaho pixels è: {winner}')

    if delta_mae > 1.0:
        print('\n→ IPOTESI CONFERMATA: training su dominio ristretto migliora accuracy.')
        print('  Possibili spiegazioni:')
        print('  - Regime climatico più omogeneo → feature più specializzate')
        print('  - Elevation #1 (Idaho) vs aspect_cos #1 (Extended): il modello Idaho')
        print('    impara la relazione elevation-SWE più precisamente nel dominio ristretto')
    elif delta_mae < -1.0:
        print('\n→ IPOTESI CONFUTATA: più dati temporali compensano il dominio più largo.')
        print('  Possibili spiegazioni:')
        print('  - 39 anni di training (1985-2024) coprono più variabilità climatica')
        print('  - Trasferimento di conoscenza da aree topograficamente simili')
    else:
        print('\n→ RISULTATO MISTO: differenza < 1mm, non conclusivo.')
        print('  I due approcci sono comparabili su Idaho pixels.')

    if xai_ext_idaho_mean is not None:
        top1_ext = CH_EXT[np.argmax(xai_ext_idaho_mean)]
        top1_idaho = xai_idaho_mean.idxmax()
        print(f'\nXAI shift:')
        print(f'  Idaho model → #1: {top1_idaho}')
        print(f'  Extended@Idaho → #1: {top1_ext}')
        if top1_idaho != top1_ext:
            print(f'  I.e. il shift {top1_idaho} → {top1_ext} suggerisce che il modello')
            print(f'  extended ha imparato una rappresentazione diversa dello spazio SWE.')
else:
    print('\n⚠️  Extended model results non disponibili. Esegui NCAR e rilancia.')

print('=' * 70)

In [ ]:
# APPENDICE: confronto con metriche dell'anno 2020 specificatamente
#
# 2020 è l'unico anno out-of-sample per ENTRAMBI i modelli → confronto più pulito

print('\n--- Anno 2020 (unico anno sovrapposto out-of-sample) ---')

row_2020 = df_idaho[df_idaho.test_year == 2020].iloc[0]
print(f'Idaho model, 2020:')
print(f'  MAE={row_2020.mae:.2f}  BIAS={row_2020.bias:+.2f}  RMSE={row_2020.rmse:.2f}  '
      f'R²={row_2020.r2:.3f}  NSE={row_2020.nse:.3f}')
print(f'  (training: 2017, 2018, 2019; val: 2021)')

if df_ext_idaho is not None:
    row_2020_ext = df_ext_idaho[df_ext_idaho.year == 2020]
    if len(row_2020_ext) > 0:
        r = row_2020_ext.iloc[0]
        print(f'\nExtended@Idaho, 2020:')
        print(f'  MAE={r[mae_col]:.2f}  BIAS={r[bias_col]:+.2f}')
        print(f'  (training: 1985-2019 escluso ogni 5 anni)')
        print(f'\nΔMAE per anno 2020: {r[mae_col] - row_2020.mae:+.2f} mm '
              f'({"Extended peggiore" if r[mae_col] > row_2020.mae else "Extended migliore"})')
    else:
        print('\nExtended@Idaho, 2020: non disponibile (NCAR pending)')